# EduVidQA test: OpenAI predict + Gemini judge

Notebook này chạy `gpt-4o-mini` trên test set EduVidQA rồi dùng Gemini chấm kết quả.

Fairness:
- Dùng cùng test scope `synthetic_test + real_world_test`.
- Prompt yêu cầu trả lời một đoạn ngắn gọn, không bullet/markdown/JSON/code fence.
- Nếu record có ảnh/frame, OpenAI sẽ nhận cả text + ảnh; nếu không có ảnh thì dùng context text.
- Gemini chấm từng câu theo cùng rubric: correctness, coverage, groundedness, FactQA Precision/Recall, clarity, critical thinking, pedagogical techniques, format, hallucination.

Notebook chỉ dùng OpenAI + Gemini, không cài Unsloth/Transformers, nên tránh lỗi Qwen3.5 dependency.

In [ ]:
# 0) Install dependencies
!pip install -q "openai" "google-genai>=1.66.0,<2.0.0" "gdown" "pandas==2.2.2" "tqdm" "pillow<12"

In [ ]:
# 1) Config + mount Drive

from getpass import getpass
from pathlib import Path
import json, time, re, os, random, base64, mimetypes
from collections import Counter, defaultdict

import pandas as pd
from tqdm.auto import tqdm
from google.colab import drive

drive.mount("/content/drive")

# ===== Main switches =====
RUN_OPENAI_PREDICT = True
RUN_GEMINI_JUDGE = True
RUN_SUMMARY = True
RUN_RANDOM_REVIEW = True

FORCE_RERUN_PREDICT = False
FORCE_REJUDGE = False

# None = full test. Set 10 for smoke test.
MAX_EVAL_SAMPLES = None
EVAL_SEED = 42

# ===== Dataset archive from previous EduVidQA pipeline =====
DATA_ARCHIVE_FILE_ID = "1uXvOVhwo8j944gRYBqpzxKZn0_EqjdzL"
DATA_ARCHIVE_FILENAME = "ft_context_vlm_clean.tar.gz"
DRIVE_DATA_CACHE = Path("/content/drive/MyDrive/eduvidqa")
DATASET_RUNTIME_ROOT = Path("/content/eduvidqa_openai_eval_dataset")
DATASET_ROOT = DATASET_RUNTIME_ROOT / "ft_context_vlm_clean"

# ===== Project paths =====
WORKDIR = Path("/content/drive/MyDrive/eduvidqa_qwen35_vl_full_r16e1")
OUTPUT_DIR = WORKDIR / "outputs"

# ===== OpenAI prediction =====
OPENAI_MODEL = "gpt-4o-mini"
OPENAI_MAX_OUTPUT_TOKENS = 256
# Request pacing. The common failure is TPM, not only RPM.
# Keep this conservative; 385 samples is small, so slower is safer.
OPENAI_RPM_LIMIT = 20
OPENAI_REQUEST_INTERVAL_SEC = 60.0 / OPENAI_RPM_LIMIT
OPENAI_MAX_RETRIES = 10
OPENAI_RETRY_FALLBACK_WAIT_SEC = 8.0
MAX_IMAGES_PER_SAMPLE = 2

OPENAI_API_KEY = getpass("OpenAI API key: ").strip()
assert OPENAI_API_KEY, "Need OpenAI API key."

# ===== Gemini judge =====
GEMINI_MODEL = "gemini-3.1-flash-lite-preview"
GEMINI_THINKING_LEVEL = "HIGH"  # MINIMAL, LOW, MEDIUM, HIGH
GEMINI_MAX_OUTPUT_TOKENS = 4096
GEMINI_RPM_LIMIT_PER_KEY = 15
GEMINI_RPD_LIMIT_PER_KEY = 500
GEMINI_RATE_LIMIT_COOLDOWN_SEC = 60
MAX_RETRIES_PER_SAMPLE = 6

GEMINI_API_KEY_1 = getpass("Gemini API key #1: ").strip()
GEMINI_API_KEY_2 = getpass("Gemini API key #2: ").strip()
GEMINI_API_KEY_3 = getpass("Gemini API key #3: ").strip()
GEMINI_API_KEYS = [GEMINI_API_KEY_1, GEMINI_API_KEY_2, GEMINI_API_KEY_3]
assert all(GEMINI_API_KEYS), "Need 3 Gemini API keys."
assert len(set(GEMINI_API_KEYS)) == len(GEMINI_API_KEYS), "Three Gemini keys should be different."

# ===== Output paths =====
RUN_TAG = f"openai_{OPENAI_MODEL}_gemini_{GEMINI_THINKING_LEVEL.lower()}_eduvidqa_test"
RUN_TAG = RUN_TAG.replace("/", "_").replace(":", "_")
EVAL_DIR = OUTPUT_DIR / "eduvidqa_openai_eval" / RUN_TAG
EVAL_DIR.mkdir(parents=True, exist_ok=True)

OPENAI_PRED_PATH = EVAL_DIR / "openai_predictions_full.jsonl"
OPENAI_JUDGE_PATH = EVAL_DIR / "openai_gemini_judge_full.jsonl"
SUMMARY_OUT = EVAL_DIR / "summary.json"
COMPARISON_CSV_OUT = EVAL_DIR / "comparison_with_existing_qwen.csv"
RANDOM_REVIEW_JSONL = EVAL_DIR / "random_review_examples.jsonl"
RANDOM_REVIEW_CSV = EVAL_DIR / "random_review_examples.csv"
RANDOM_REVIEW_MD = EVAL_DIR / "random_review_examples.md"

# Optional: compare against existing Qwen baseline/FT unified judge if files exist.
OPTIONAL_BASELINE_JUDGE_PATH = (
    OUTPUT_DIR / "eval_official_eduvidqa_metrics"
    / "gemini_gemini-3.1-flash-lite-preview_thinking_high_unified_1call"
    / "baseline_unified_judge_full.jsonl"
)
OPTIONAL_FT_JUDGE_PATH = (
    OUTPUT_DIR / "eval_official_eduvidqa_metrics"
    / "gemini_gemini-3.1-flash-lite-preview_thinking_high_unified_1call"
    / "fine_tuned_unified_judge_full.jsonl"
)

print("Eval dir:", EVAL_DIR)
print("OpenAI prediction:", OPENAI_PRED_PATH)
print("OpenAI judge:", OPENAI_JUDGE_PATH)
print("Optional baseline judge exists:", OPTIONAL_BASELINE_JUDGE_PATH.exists(), OPTIONAL_BASELINE_JUDGE_PATH)
print("Optional FT judge exists:", OPTIONAL_FT_JUDGE_PATH.exists(), OPTIONAL_FT_JUDGE_PATH)

In [ ]:
# 2) Utilities

def read_jsonl(path):
    path = Path(path)
    rows = []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def read_jsonl_safe(path):
    path = Path(path)
    rows, bad = [], []
    if not path.exists():
        return rows
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            raw = line.strip()
            if not raw:
                continue
            try:
                rows.append(json.loads(raw))
            except json.JSONDecodeError as exc:
                bad.append({"line_no": line_no, "error": str(exc), "prefix": raw[:300]})
    if bad:
        bad_path = path.with_suffix(path.suffix + ".bad_lines.json")
        write_json(bad_path, bad)
        print(f"[WARN] skipped {len(bad)} bad lines in {path}; details -> {bad_path}")
    return rows

def append_jsonl(path, row):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

def overwrite_jsonl(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

def read_json(path, default=None):
    path = Path(path)
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding="utf-8"))

def normalize_text(s):
    return re.sub(r"\s+", " ", str(s or "")).strip()

def get_id(row):
    return row.get("id") or row.get("eval_id") or row.get("item_id") or row.get("qid")

def recursive_text_values(obj, skip_keys=None, max_items=80):
    skip_keys = set(skip_keys or [])
    vals = []
    def visit(x, path=""):
        if len(vals) >= max_items:
            return
        if isinstance(x, dict):
            for k, v in x.items():
                lk = str(k).lower()
                if lk in skip_keys:
                    continue
                visit(v, f"{path}.{k}" if path else str(k))
        elif isinstance(x, list):
            for v in x[:20]:
                visit(v, path)
        elif isinstance(x, str):
            s = x.strip()
            if len(s) >= 20:
                vals.append((path, s))
    visit(obj)
    return vals

def get_question(row):
    for key in ["question", "query", "prompt", "user_question"]:
        if isinstance(row.get(key), str) and row.get(key).strip():
            return normalize_text(row[key])
    for container_key in ["input", "inputs", "data"]:
        container = row.get(container_key)
        if isinstance(container, dict):
            for key in ["question", "query", "prompt", "user_question"]:
                if isinstance(container.get(key), str) and container.get(key).strip():
                    return normalize_text(container[key])
    msgs = row.get("messages")
    if isinstance(msgs, list):
        for m in msgs:
            if isinstance(m, dict) and m.get("role") == "user":
                content = m.get("content")
                if isinstance(content, str):
                    return normalize_text(content[:1200])
                if isinstance(content, list):
                    texts = [c.get("text", "") for c in content if isinstance(c, dict) and c.get("type") == "text"]
                    if texts:
                        return normalize_text(" ".join(texts)[:1200])
    if isinstance(row.get("text_input"), str):
        txt = row["text_input"]
        parts = [p.strip() for p in re.split(r"\n+", txt) if p.strip()]
        question_like = [p for p in parts if "?" in p]
        if question_like:
            return normalize_text(question_like[-1])
        return normalize_text(txt[:800])
    return ""

def get_reference_answer(row):
    for key in ["answer", "reference_answer", "gold_answer", "target", "label", "output"]:
        if isinstance(row.get(key), str) and row.get(key).strip():
            return normalize_text(row[key])
    for container_key in ["reference", "labels", "target", "output"]:
        container = row.get(container_key)
        if isinstance(container, dict):
            parts = []
            for key in ["answer_text", "answer", "reference_answer", "gold_answer", "explanation", "target"]:
                if isinstance(container.get(key), str) and container.get(key).strip():
                    parts.append(container[key])
            if parts:
                return normalize_text(" ".join(parts))
    msgs = row.get("messages")
    if isinstance(msgs, list):
        for m in reversed(msgs):
            if isinstance(m, dict) and m.get("role") in ["assistant", "model"]:
                content = m.get("content")
                if isinstance(content, str):
                    return normalize_text(content)
    return ""

def get_context(row, max_chars=7000):
    pieces = []
    for key in ["context", "text_input", "transcript_context", "transcript", "transcript_window", "source_evidence_span"]:
        val = row.get(key)
        if isinstance(val, str) and val.strip():
            pieces.append(f"{key}: {val.strip()}")
        elif isinstance(val, dict):
            texts = recursive_text_values(val, skip_keys={"answer", "reference_answer", "gold_answer", "target"})
            if texts:
                pieces.append(f"{key}: " + "\n".join(f"{p}: {t}" for p, t in texts[:20]))
    for container_key in ["input", "inputs", "metadata"]:
        container = row.get(container_key)
        if isinstance(container, dict):
            texts = recursive_text_values(
                container,
                skip_keys={"question", "answer", "reference_answer", "gold_answer", "target", "label"},
                max_items=40,
            )
            if texts:
                pieces.append(f"{container_key}: " + "\n".join(f"{p}: {t}" for p, t in texts[:25]))
    text = "\n\n".join(pieces)
    if not text:
        text = json.dumps(row, ensure_ascii=False)[:max_chars]
    if len(text) > max_chars:
        text = text[:max_chars//2] + "\n...[TRUNCATED]...\n" + text[-max_chars//2:]
    return text

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

def find_image_paths(row, max_images=MAX_IMAGES_PER_SAMPLE):
    paths = []
    def add_candidate(s):
        if not isinstance(s, str):
            return
        if Path(s).suffix.lower() not in IMAGE_EXTS:
            return
        candidates = [Path(s), DATASET_ROOT / s, DATASET_RUNTIME_ROOT / s, Path("/content") / s.lstrip("/")]
        for p in candidates:
            if p.exists() and p.is_file():
                paths.append(p)
                break
    def visit(x):
        if len(paths) >= max_images:
            return
        if isinstance(x, dict):
            for v in x.values():
                visit(v)
        elif isinstance(x, list):
            for v in x:
                visit(v)
        elif isinstance(x, str):
            add_candidate(x)
    visit(row)
    seen, out = set(), []
    for p in paths:
        rp = str(p.resolve())
        if rp not in seen:
            seen.add(rp)
            out.append(p)
        if len(out) >= max_images:
            break
    return out

def image_to_data_url(path):
    path = Path(path)
    mime = mimetypes.guess_type(str(path))[0] or "image/jpeg"
    data = base64.b64encode(path.read_bytes()).decode("utf-8")
    return f"data:{mime};base64,{data}"

def build_answer_prompt(row):
    return f"""You are answering an educational video question using only the provided context and images.

Requirements:
- Answer in the same language as the question/reference.
- Answer in one concise paragraph.
- Do not use bullet points, markdown tables, JSON, or code fences.
- Do not reveal hidden reasoning.
- If the context is insufficient, say what can be answered from the context instead of inventing details.

Context:
{get_context(row)}

Question:
{get_question(row)}

Answer:"""

In [ ]:
# 3) Download/extract EduVidQA test data

DRIVE_DATA_CACHE.mkdir(parents=True, exist_ok=True)
archive_path = DRIVE_DATA_CACHE / DATA_ARCHIVE_FILENAME

if not archive_path.exists():
    print("Dataset archive not found on Drive; downloading with gdown...")
    !gdown {DATA_ARCHIVE_FILE_ID} -O {archive_path}

assert archive_path.exists(), f"Dataset archive not found: {archive_path}"

if not (DATASET_ROOT / "synthetic_test.jsonl").exists():
    print("Extracting dataset archive...")
    DATASET_RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
    import tarfile
    with tarfile.open(archive_path, "r:gz") as archive:
        archive.extractall(DATASET_RUNTIME_ROOT)
else:
    print("Dataset already extracted:", DATASET_ROOT)

synthetic_test = read_jsonl_safe(DATASET_ROOT / "synthetic_test.jsonl")
real_world_test = read_jsonl_safe(DATASET_ROOT / "real_world_test.jsonl")

for r in synthetic_test:
    r.setdefault("split", "synthetic_test")
for r in real_world_test:
    r.setdefault("split", "real_world_test")

test_rows = synthetic_test + real_world_test

if MAX_EVAL_SAMPLES is not None:
    rng = random.Random(EVAL_SEED)
    test_rows = rng.sample(test_rows, k=min(MAX_EVAL_SAMPLES, len(test_rows)))

bad = []
for i, r in enumerate(test_rows):
    if not get_id(r):
        bad.append((i, "missing_id"))
    if not get_question(r):
        bad.append((i, "missing_question"))
    if not get_reference_answer(r):
        bad.append((i, "missing_reference_answer"))

print("synthetic_test:", len(synthetic_test))
print("real_world_test:", len(real_world_test))
print("used rows:", len(test_rows))
print("split counts:", dict(Counter(r.get("split", "unknown") for r in test_rows)))
print("schema issues:", len(bad))
if bad[:5]:
    print("sample schema issues:", bad[:5])

assert not bad, "Some rows miss id/question/reference. Inspect bad list."

print("First id:", get_id(test_rows[0]))
print("First question:", get_question(test_rows[0])[:300])
print("First reference:", get_reference_answer(test_rows[0])[:300])
print("First image paths:", [str(p) for p in find_image_paths(test_rows[0])])

In [ ]:

# 4) OpenAI prediction on EduVidQA test set
# v2: robust retry/backoff for TPM/RPM 429 errors.

from openai import OpenAI, RateLimitError, APIConnectionError, APIStatusError
openai_client = OpenAI(api_key=OPENAI_API_KEY)

def parse_openai_retry_after_seconds(exc):
    """Parse messages like 'Please try again in 489ms' or 'try again in 2s'."""
    msg = str(exc)
    m = re.search(r"try again in\s*([0-9.]+)\s*(ms|milliseconds|s|sec|secs|second|seconds)", msg, flags=re.I)
    if not m:
        return None
    value = float(m.group(1))
    unit = m.group(2).lower()
    if unit.startswith("ms") or unit.startswith("millisecond"):
        return max(0.25, value / 1000.0)
    return max(0.25, value)

def is_retryable_openai_error(exc):
    msg = str(exc).lower()
    if isinstance(exc, (RateLimitError, APIConnectionError)):
        return True
    if isinstance(exc, APIStatusError) and getattr(exc, "status_code", None) in [408, 409, 429, 500, 502, 503, 504]:
        return True
    return "rate limit" in msg or "tokens per min" in msg or "too many requests" in msg or "temporarily unavailable" in msg

def latest_rows_by_id(rows):
    """Keep only the latest row for each id; useful after a previous run wrote failed duplicates."""
    latest = {}
    order = []
    for r in rows:
        rid = r.get("id")
        if rid is None:
            continue
        if rid not in latest:
            order.append(rid)
        latest[rid] = r
    return [latest[rid] for rid in order]

def openai_predict_one(row):
    prompt = build_answer_prompt(row)
    content = [{"type": "text", "text": prompt}]

    image_paths = find_image_paths(row)
    for p in image_paths:
        content.append({"type": "image_url", "image_url": {"url": image_to_data_url(p)}})

    last_exc = None
    for attempt in range(1, OPENAI_MAX_RETRIES + 1):
        try:
            resp = openai_client.chat.completions.create(
                model=OPENAI_MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You answer educational video questions using only the supplied context and images. "
                            "Return one concise paragraph and avoid markdown/bullets/JSON."
                        ),
                    },
                    {"role": "user", "content": content},
                ],
                temperature=0,
                max_tokens=OPENAI_MAX_OUTPUT_TOKENS,
            )

            usage = getattr(resp, "usage", None)
            usage_dict = {
                "prompt_tokens": int(getattr(usage, "prompt_tokens", 0) or 0) if usage else 0,
                "completion_tokens": int(getattr(usage, "completion_tokens", 0) or 0) if usage else 0,
                "total_tokens": int(getattr(usage, "total_tokens", 0) or 0) if usage else 0,
            }
            return normalize_text(resp.choices[0].message.content or ""), usage_dict, [str(p) for p in image_paths]

        except Exception as exc:
            last_exc = exc
            if not is_retryable_openai_error(exc):
                raise

            retry_after = parse_openai_retry_after_seconds(exc)
            if retry_after is None:
                retry_after = min(90.0, OPENAI_RETRY_FALLBACK_WAIT_SEC * attempt)

            # Add a small safety buffer because TPM is sliding-window based.
            wait = retry_after + 1.0 + random.uniform(0.0, 0.75)
            print(f"[OpenAI retry] attempt={attempt}/{OPENAI_MAX_RETRIES}; wait={wait:.2f}s; error={str(exc)[:220]}")
            time.sleep(wait)

    raise RuntimeError(f"OpenAI prediction failed after {OPENAI_MAX_RETRIES} retries: {last_exc}")

def run_openai_predictions():
    existing = [] if FORCE_RERUN_PREDICT else read_jsonl_safe(OPENAI_PRED_PATH)
    rows = latest_rows_by_id(existing)

    # Clean duplicate failed rows left from earlier versions.
    if existing and len(rows) != len(existing) and not FORCE_RERUN_PREDICT:
        print(f"Compacting prediction file: {len(existing)} rows -> {len(rows)} latest rows")
        overwrite_jsonl(OPENAI_PRED_PATH, rows)

    ok_ids = {r.get("id") for r in rows if r.get("status") == "ok"}
    if len(ok_ids) >= len(test_rows):
        print("OpenAI predictions already complete:", OPENAI_PRED_PATH)
        return rows

    last_call = 0.0

    for row in tqdm(test_rows, desc="OpenAI predict", unit="sample"):
        rid = get_id(row)
        if rid in ok_ids:
            continue

        # RPM pacing before each new sample. Retry logic handles TPM/429 inside openai_predict_one.
        wait = max(0.0, OPENAI_REQUEST_INTERVAL_SEC - (time.monotonic() - last_call))
        if wait:
            time.sleep(wait)

        t0 = time.time()
        try:
            pred, usage, image_paths = openai_predict_one(row)
            status, error = "ok", None
        except Exception as exc:
            pred = ""
            usage = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
            image_paths = []
            status, error = "failed", str(exc)[:1000]
            print("[WARN] OpenAI prediction failed after retries:", rid, error)

        last_call = time.monotonic()
        out = {
            "id": rid,
            "split": row.get("split"),
            "model": OPENAI_MODEL,
            "question": get_question(row),
            "reference_answer": get_reference_answer(row),
            "prediction": pred,
            "image_paths": image_paths,
            "latency_sec": time.time() - t0,
            "status": status,
            "error": error,
            **usage,
        }

        append_jsonl(OPENAI_PRED_PATH, out)

        # Keep in-memory rows latest-by-id.
        rows = [r for r in rows if r.get("id") != rid]
        rows.append(out)

        if status == "ok":
            ok_ids.add(rid)

    # Final compaction after successful resume.
    rows = latest_rows_by_id(read_jsonl_safe(OPENAI_PRED_PATH))
    overwrite_jsonl(OPENAI_PRED_PATH, rows)
    return rows

if RUN_OPENAI_PREDICT:
    openai_predictions = run_openai_predictions()
else:
    openai_predictions = latest_rows_by_id(read_jsonl_safe(OPENAI_PRED_PATH))

print("OpenAI predictions:", len(openai_predictions), OPENAI_PRED_PATH)
print("ok predictions:", sum(1 for r in openai_predictions if r.get("status") == "ok"))
print("failed predictions:", sum(1 for r in openai_predictions if r.get("status") != "ok"))
print("OpenAI token usage:", {
    "prompt_tokens": sum(int(r.get("prompt_tokens", 0) or 0) for r in openai_predictions),
    "completion_tokens": sum(int(r.get("completion_tokens", 0) or 0) for r in openai_predictions),
    "total_tokens": sum(int(r.get("total_tokens", 0) or 0) for r in openai_predictions),
})


In [ ]:
# 5) Local format metrics

def pred_answer(row):
    return normalize_text(row.get("prediction", ""))

def compute_rule_metrics(pred_rows):
    answers = [pred_answer(r) for r in pred_rows if r.get("status") == "ok"]
    n = len(answers) or 1
    def has_bullets(a):
        return bool(re.search(r"(?m)^\s*([-*]|\d+[.)])\s+", a))
    def has_code_or_json(a):
        s = a.strip()
        return "```" in s or s.startswith("{") or s.startswith("[") or s.lower().startswith("json")
    def one_paragraph(a):
        return "\n" not in a.strip()
    def no_think(a):
        low = a.lower()
        return "<think>" not in low and "</think>" not in low
    def format_pass(a):
        return bool(a.strip()) and no_think(a) and one_paragraph(a) and not has_bullets(a) and not has_code_or_json(a)
    word_counts = [len(a.split()) for a in answers]
    return {
        "sample_count": len(pred_rows),
        "ok_count": sum(1 for r in pred_rows if r.get("status") == "ok"),
        "format_pass_rate": sum(format_pass(a) for a in answers) / n,
        "no_think_rate": sum(no_think(a) for a in answers) / n,
        "one_paragraph_rate": sum(one_paragraph(a) for a in answers) / n,
        "markdown_bullet_rate": sum(has_bullets(a) for a in answers) / n,
        "json_or_code_fence_rate": sum(has_code_or_json(a) for a in answers) / n,
        "empty_answer_rate": sum(not a.strip() for a in answers) / n,
        "answer_word_count_mean": sum(word_counts) / len(word_counts) if word_counts else None,
        "answer_word_count_p95": sorted(word_counts)[int(0.95 * (len(word_counts)-1))] if word_counts else None,
        "latency_sec_mean": sum(float(r.get("latency_sec", 0) or 0) for r in pred_rows) / max(1, len(pred_rows)),
    }

openai_rule_metrics = compute_rule_metrics(openai_predictions)
print(json.dumps(openai_rule_metrics, ensure_ascii=False, indent=2))

In [ ]:
# 6) Gemini round-robin judge

from google import genai
from google.genai import types

JUDGE_SCHEMA = {
    "type": "object",
    "properties": {
        "factqa_precision_score": {"type": "string", "description": "Fraction '<supported>/<total>' for reference claims supported by prediction."},
        "factqa_recall_score": {"type": "string", "description": "Fraction '<supported>/<total>' for prediction claims supported by reference."},
        "correctness": {"type": "integer", "minimum": 1, "maximum": 5},
        "coverage": {"type": "integer", "minimum": 1, "maximum": 5},
        "groundedness": {"type": "integer", "minimum": 1, "maximum": 5},
        "clarity": {"type": "integer", "minimum": 1, "maximum": 5},
        "critical_thinking": {"type": "integer", "minimum": 1, "maximum": 5},
        "pedagogical_techniques": {"type": "integer", "minimum": 1, "maximum": 5},
        "format": {"type": "integer", "minimum": 1, "maximum": 5},
        "hallucination": {"type": "boolean"},
        "rationale": {"type": "string", "description": "Short Vietnamese explanation under 80 words."}
    },
    "required": ["factqa_precision_score", "factqa_recall_score", "correctness", "coverage", "groundedness", "clarity", "critical_thinking", "pedagogical_techniques", "format", "hallucination", "rationale"]
}

def get_thinking_level(name):
    name = str(name or "HIGH").upper()
    valid = {"MINIMAL", "LOW", "MEDIUM", "HIGH", "THINKING_LEVEL_UNSPECIFIED"}
    if name not in valid:
        raise ValueError(f"Unsupported thinking level: {name}")
    return getattr(types.ThinkingLevel, name)

def is_rate_limit_error(exc):
    code = getattr(exc, "code", None)
    msg = str(getattr(exc, "message", "") or exc).lower()
    return code == 429 or "rate limit" in msg or "quota" in msg or "resource exhausted" in msg or "high demand" in msg

def extract_json_obj(text):
    text = str(text or "").strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE).strip()
    text = re.sub(r"\s*```$", "", text).strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        try:
            return json.loads(text[start:end+1])
        except Exception:
            pass
    return None

class GeminiRoundRobin:
    def __init__(self, keys, model_name, rpm=15, rpd=500, cooldown=60):
        self.clients = [genai.Client(api_key=k) for k in keys]
        self.model_name = model_name
        self.rpm = rpm
        self.rpd = rpd
        self.cooldown = cooldown
        self.next_idx = 0
        self.calls = [0 for _ in keys]
        self.rate_limit_errors = [0 for _ in keys]
        self.rate_limited_until = [0.0 for _ in keys]
        self.last_call = [0.0 for _ in keys]
        self.usage = {"prompt_token_count": 0, "candidates_token_count": 0, "thoughts_token_count": 0, "total_token_count": 0}

    def _take_slot(self):
        now = time.monotonic()
        for _ in range(len(self.clients)):
            idx = self.next_idx
            self.next_idx = (self.next_idx + 1) % len(self.clients)
            if self.calls[idx] >= self.rpd:
                continue
            if self.rate_limited_until[idx] > now:
                continue
            min_gap = 60.0 / max(1, self.rpm)
            wait = max(0.0, min_gap - (now - self.last_call[idx]))
            if wait:
                time.sleep(wait)
            self.calls[idx] += 1
            self.last_call[idx] = time.monotonic()
            return idx, self.clients[idx]
        wait_until = min(self.rate_limited_until) if self.rate_limited_until else time.monotonic() + self.cooldown
        wait = max(5.0, wait_until - time.monotonic())
        print(f"All Gemini slots cooling down; sleeping {wait:.1f}s")
        time.sleep(wait)
        return self._take_slot()

    def _mark_rate_limited(self, idx, exc):
        self.rate_limit_errors[idx] += 1
        self.rate_limited_until[idx] = time.monotonic() + self.cooldown
        print(f"Gemini slot {idx} rate/quota/high-demand; switching. Error: {str(exc)[:250]}")

    def _record_usage(self, response):
        usage = getattr(response, "usage_metadata", None)
        if usage is None:
            return
        for k in self.usage:
            self.usage[k] += int(getattr(usage, k, 0) or 0)

    def call_json(self, prompt, schema, max_retries=None):
        max_retries = max_retries or MAX_RETRIES_PER_SAMPLE
        last_exc = None
        for attempt in range(1, max_retries + 1):
            idx, client = self._take_slot()
            try:
                config_kwargs = dict(
                    temperature=0,
                    max_output_tokens=GEMINI_MAX_OUTPUT_TOKENS,
                    response_mime_type="application/json",
                    thinking_config=types.ThinkingConfig(
                        thinking_level=get_thinking_level(GEMINI_THINKING_LEVEL)
                    ),
                )
                try:
                    config_kwargs["response_schema"] = schema
                    config = types.GenerateContentConfig(**config_kwargs)
                except TypeError:
                    config_kwargs.pop("response_schema", None)
                    config = types.GenerateContentConfig(**config_kwargs)
                resp = client.models.generate_content(model=self.model_name, contents=prompt, config=config)
                self._record_usage(resp)
                text = getattr(resp, "text", None) or str(resp)
                obj = extract_json_obj(text)
                if isinstance(obj, dict):
                    return obj
                raise ValueError(f"Cannot parse JSON. Prefix={text[:500]!r}")
            except Exception as exc:
                last_exc = exc
                if is_rate_limit_error(exc):
                    self._mark_rate_limited(idx, exc)
                    continue
                print(f"Gemini judge error attempt={attempt}: {str(exc)[:300]}")
                time.sleep(min(2 * attempt, 20))
        raise RuntimeError(f"Gemini judge failed after {max_retries} attempts: {last_exc}")

gemini_judge = GeminiRoundRobin(GEMINI_API_KEYS, GEMINI_MODEL, rpm=GEMINI_RPM_LIMIT_PER_KEY, rpd=GEMINI_RPD_LIMIT_PER_KEY, cooldown=GEMINI_RATE_LIMIT_COOLDOWN_SEC)

def parse_fraction_score(score_str):
    m = re.search(r"(\d+)\s*/\s*(\d+)", str(score_str or ""))
    if not m:
        return None
    num, den = int(m.group(1)), int(m.group(2))
    if den <= 0:
        return None
    return num / den

def composite_score(j):
    score = (
        0.30 * float(j.get("correctness", 0) or 0) / 5
        + 0.20 * float(j.get("coverage", 0) or 0) / 5
        + 0.20 * float(j.get("groundedness", 0) or 0) / 5
        + 0.10 * float(j.get("clarity", 0) or 0) / 5
        + 0.10 * float(j.get("format", 0) or 0) / 5
        + 0.05 * float(j.get("pedagogical_techniques", 0) or 0) / 5
        + 0.05 * float(j.get("critical_thinking", 0) or 0) / 5
    )
    if bool(j.get("hallucination", False)):
        score -= 0.15
    return max(0.0, min(1.0, score))

def build_judge_prompt(row, pred_row):
    return f"""Bạn là evaluator nghiêm ngặt cho bài hỏi đáp video giáo dục.

Chấm prediction theo reference answer và context. Trả về JSON đúng schema.

Rubric:
- correctness: đúng với reference/label, thang 1-5.
- coverage: phủ đủ các ý quan trọng trong reference, thang 1-5.
- groundedness: được context hỗ trợ, không bịa ngoài context, thang 1-5.
- clarity: rõ ràng, dễ hiểu, thang 1-5.
- critical_thinking: có gợi mở tư duy/so sánh/khám phá, thang 1-5.
- pedagogical_techniques: có ví dụ, giải thích từng bước, analogy hoặc hỗ trợ học tập, thang 1-5.
- format: một đoạn ngắn gọn, không bullet, không markdown, không JSON/code fence, thang 1-5.
- hallucination: true nếu có thông tin không được context/reference hỗ trợ.
- FactQA Precision: atomic claims trong reference được prediction support, dạng "<supported>/<total>".
- FactQA Recall: atomic claims trong prediction được reference support, dạng "<supported>/<total>".

Context:
{get_context(row)}

Question:
{get_question(row)}

Reference answer:
{get_reference_answer(row)}

Prediction:
{pred_row.get("prediction", "")}

Chỉ trả JSON. Không thêm markdown.
""".strip()

In [ ]:
# 7) Run Gemini judge on OpenAI predictions

def run_gemini_judge():
    pred_map = {r.get("id"): r for r in openai_predictions if r.get("status") == "ok"}
    if FORCE_REJUDGE and OPENAI_JUDGE_PATH.exists():
        backup = OPENAI_JUDGE_PATH.with_suffix(OPENAI_JUDGE_PATH.suffix + f".backup_{int(time.time())}")
        OPENAI_JUDGE_PATH.rename(backup)
        print("Backed up old judge file:", backup)
    judged = [] if FORCE_REJUDGE else read_jsonl_safe(OPENAI_JUDGE_PATH)
    done_ids = {r.get("id") for r in judged if r.get("judge_status") == "ok"}
    print("Existing judged OK:", len(done_ids), "/", len(test_rows))
    print("Judge output:", OPENAI_JUDGE_PATH)
    for row in tqdm(test_rows, desc="Gemini judge OpenAI", unit="sample"):
        rid = get_id(row)
        if rid in done_ids:
            continue
        pred_row = pred_map.get(rid)
        if pred_row is None:
            out = {"id": rid, "split": row.get("split"), "judge_status": "missing_prediction"}
            append_jsonl(OPENAI_JUDGE_PATH, out)
            judged.append(out)
            continue
        prompt = build_judge_prompt(row, pred_row)
        try:
            obj = gemini_judge.call_json(prompt, JUDGE_SCHEMA)
            fqa_p = parse_fraction_score(obj.get("factqa_precision_score"))
            fqa_r = parse_fraction_score(obj.get("factqa_recall_score"))
            if fqa_p is None:
                fqa_p = 0.0
            if fqa_r is None:
                fqa_r = 0.0
            out = {
                "id": rid,
                "split": row.get("split"),
                "judge_status": "ok",
                "model": OPENAI_MODEL,
                "judge_model": GEMINI_MODEL,
                "gemini_thinking_level": GEMINI_THINKING_LEVEL,
                "metric_protocol": "gemini_contextual_openqa_eval",
                "factqa_precision": fqa_p,
                "factqa_recall": fqa_r,
                "factqa_precision_score_raw": obj.get("factqa_precision_score"),
                "factqa_recall_score_raw": obj.get("factqa_recall_score"),
                "correctness": int(obj.get("correctness", 1)),
                "coverage": int(obj.get("coverage", 1)),
                "groundedness": int(obj.get("groundedness", 1)),
                "clarity": int(obj.get("clarity", 1)),
                "critical_thinking": int(obj.get("critical_thinking", 1)),
                "pedagogical_techniques": int(obj.get("pedagogical_techniques", 1)),
                "format": int(obj.get("format", 1)),
                "hallucination": bool(obj.get("hallucination", False)),
                "rationale": obj.get("rationale", ""),
            }
            out["score"] = composite_score(out)
            out["hallucination_proxy_1_minus_factqa_precision"] = 1.0 - fqa_p
        except Exception as exc:
            out = {"id": rid, "split": row.get("split"), "judge_status": "failed", "judge_error": str(exc)[:1000]}
            print("[WARN] judge failed:", rid, exc)
        append_jsonl(OPENAI_JUDGE_PATH, out)
        judged.append(out)
        if out.get("judge_status") == "ok":
            done_ids.add(rid)
    return read_jsonl_safe(OPENAI_JUDGE_PATH)

if RUN_GEMINI_JUDGE:
    openai_judged = run_gemini_judge()
else:
    openai_judged = read_jsonl_safe(OPENAI_JUDGE_PATH)

print("Judged rows:", len(openai_judged))
print("OK:", sum(1 for r in openai_judged if r.get("judge_status") == "ok"))
print("Gemini calls:", gemini_judge.calls)
print("Gemini usage:", json.dumps(gemini_judge.usage, ensure_ascii=False, indent=2))

In [ ]:
# 8) Summary and optional comparison with existing Qwen judge files

NUMERIC_JUDGE_FIELDS = [
    "score", "factqa_precision", "factqa_recall", "correctness", "coverage", "groundedness",
    "clarity", "critical_thinking", "pedagogical_techniques", "format",
    "hallucination_proxy_1_minus_factqa_precision",
]

def ok_judged(rows):
    return [r for r in rows if r.get("judge_status") == "ok"]

def mean(vals):
    vals = [v for v in vals if v is not None]
    return sum(vals) / len(vals) if vals else None

def aggregate_judge(rows):
    rows = ok_judged(rows)
    out = {"sample_count": len(rows)}
    for f in NUMERIC_JUDGE_FIELDS:
        vals = []
        for r in rows:
            if f in r:
                try:
                    vals.append(float(r[f]))
                except Exception:
                    pass
        out[f + "_mean"] = mean(vals)
        out[f + "_count"] = len(vals)
    halluc = [1.0 if r.get("hallucination") else 0.0 for r in rows if "hallucination" in r]
    out["hallucination_rate"] = mean(halluc)
    out["by_split"] = {}
    for split in sorted(set(r.get("split", "unknown") for r in rows)):
        sub = [r for r in rows if r.get("split", "unknown") == split]
        split_out = {"sample_count": len(sub)}
        for f in NUMERIC_JUDGE_FIELDS:
            vals = []
            for r in sub:
                if f in r:
                    try:
                        vals.append(float(r[f]))
                    except Exception:
                        pass
            split_out[f + "_mean"] = mean(vals)
        split_out["hallucination_rate"] = mean([1.0 if r.get("hallucination") else 0.0 for r in sub if "hallucination" in r])
        out["by_split"][split] = split_out
    return out

def adapt_existing_qwen_metrics(rows, model_name):
    ok = ok_judged(rows)
    out = {"model": model_name, "sample_count": len(ok)}
    fields = ["entailment_score", "factqa_precision", "factqa_recall", "clarity", "critical_thinking", "pedagogical_techniques", "hallucination_proxy_1_minus_factqa_precision"]
    for f in fields:
        vals = []
        for r in ok:
            if f in r:
                try:
                    vals.append(float(r[f]))
                except Exception:
                    pass
        out[f + "_mean"] = mean(vals)
    return out

openai_judge_metrics = aggregate_judge(openai_judged)
existing_baseline = read_jsonl_safe(OPTIONAL_BASELINE_JUDGE_PATH)
existing_ft = read_jsonl_safe(OPTIONAL_FT_JUDGE_PATH)

comparison_rows = []
for k, v in openai_judge_metrics.items():
    if isinstance(v, (int, float)) or v is None:
        comparison_rows.append({"model": OPENAI_MODEL, "metric": k, "value": v, "protocol": "gemini_contextual_openqa_eval"})

if existing_baseline:
    b = adapt_existing_qwen_metrics(existing_baseline, "qwen_base")
    for k, v in b.items():
        if k != "model":
            comparison_rows.append({"model": "qwen_base", "metric": k, "value": v, "protocol": "existing_unified_qwen_judge"})
if existing_ft:
    f = adapt_existing_qwen_metrics(existing_ft, "qwen_ft")
    for k, v in f.items():
        if k != "model":
            comparison_rows.append({"model": "qwen_ft", "metric": k, "value": v, "protocol": "existing_unified_qwen_judge"})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(COMPARISON_CSV_OUT, index=False)

summary = {
    "config": {
        "dataset_archive": str(archive_path),
        "dataset_root": str(DATASET_ROOT),
        "openai_model": OPENAI_MODEL,
        "openai_max_output_tokens": OPENAI_MAX_OUTPUT_TOKENS,
        "gemini_model": GEMINI_MODEL,
        "gemini_thinking_level": GEMINI_THINKING_LEVEL,
        "max_eval_samples": MAX_EVAL_SAMPLES,
        "gemini_calls": gemini_judge.calls,
        "gemini_usage": gemini_judge.usage,
    },
    "paths": {
        "openai_predictions": str(OPENAI_PRED_PATH),
        "openai_judge": str(OPENAI_JUDGE_PATH),
        "summary": str(SUMMARY_OUT),
        "comparison_csv": str(COMPARISON_CSV_OUT),
        "optional_baseline_judge": str(OPTIONAL_BASELINE_JUDGE_PATH),
        "optional_ft_judge": str(OPTIONAL_FT_JUDGE_PATH),
    },
    "rule_metrics": {"openai": openai_rule_metrics},
    "judge_metrics": {"openai": openai_judge_metrics},
    "optional_existing_qwen": {
        "baseline_found": bool(existing_baseline),
        "ft_found": bool(existing_ft),
        "baseline_metrics": adapt_existing_qwen_metrics(existing_baseline, "qwen_base") if existing_baseline else None,
        "ft_metrics": adapt_existing_qwen_metrics(existing_ft, "qwen_ft") if existing_ft else None,
    },
}

write_json(SUMMARY_OUT, summary)

print(json.dumps(summary, ensure_ascii=False, indent=2)[:6000])
print("Saved summary:", SUMMARY_OUT)
print("Saved comparison:", COMPARISON_CSV_OUT)
display(comparison_df.head(80))

In [ ]:
# 9) Random qualitative review examples

if RUN_RANDOM_REVIEW:
    pred_map = {r.get("id"): r for r in openai_predictions if r.get("status") == "ok"}
    row_map = {get_id(r): r for r in test_rows}
    ok_rows = ok_judged(openai_judged)
    rng = random.Random(EVAL_SEED)
    sample = rng.sample(ok_rows, k=min(8, len(ok_rows)))
    review_rows = []
    for j in sample:
        rid = j.get("id")
        src = row_map[rid]
        pred = pred_map[rid]
        review_rows.append({
            "id": rid,
            "split": j.get("split"),
            "question": get_question(src),
            "reference_answer": get_reference_answer(src),
            "openai_answer": pred.get("prediction"),
            "score": j.get("score"),
            "correctness": j.get("correctness"),
            "coverage": j.get("coverage"),
            "groundedness": j.get("groundedness"),
            "clarity": j.get("clarity"),
            "format": j.get("format"),
            "hallucination": j.get("hallucination"),
            "factqa_precision": j.get("factqa_precision"),
            "factqa_recall": j.get("factqa_recall"),
            "rationale": j.get("rationale"),
        })

    overwrite_jsonl(RANDOM_REVIEW_JSONL, review_rows)
    review_df = pd.DataFrame(review_rows)
    review_df.to_csv(RANDOM_REVIEW_CSV, index=False)

    md = []
    for r in review_rows:
        md.append(f"## {r['id']} | split={r['split']} | score={r['score']}\n")
        md.append(f"**Câu hỏi**\n\n{r['question']}\n")
        md.append(f"**Reference**\n\n{r['reference_answer']}\n")
        md.append(f"**OpenAI answer**\n\n{r['openai_answer']}\n")
        md.append("**Gemini judge**\n\n")
        md.append(
            f"- correctness={r['correctness']}, coverage={r['coverage']}, groundedness={r['groundedness']}, "
            f"clarity={r['clarity']}, format={r['format']}, hallucination={r['hallucination']}\n"
            f"- FactQA-P={r['factqa_precision']}, FactQA-R={r['factqa_recall']}\n"
            f"- Lý do: {r['rationale']}\n"
        )
        md.append("\n---\n")

    RANDOM_REVIEW_MD.write_text("\n".join(md), encoding="utf-8")
    display(review_df)
    print("Saved:", RANDOM_REVIEW_MD)

In [ ]:
# 10) Status
print("Eval dir:", EVAL_DIR)
print("Prediction:", OPENAI_PRED_PATH, OPENAI_PRED_PATH.exists())
print("Judge:", OPENAI_JUDGE_PATH, OPENAI_JUDGE_PATH.exists())
print("Summary:", SUMMARY_OUT, SUMMARY_OUT.exists())
print("Comparison:", COMPARISON_CSV_OUT, COMPARISON_CSV_OUT.exists())